[![Colab Badge](https://img.shields.io/badge/Open_in_Colab-blue?style=for-the-badge)][colab-link]
<a href="javascript:void(0);" onclick="openJupyterWidget('https://github.com/nmfs-opensci/nmfshackdays-2025/blob/main/topics-2025/2025-dask/Dask_gateway.ipynb');">
    <img src="https://img.shields.io/badge/Open_in_JupyterHub-orange?style=for-the-badge" alt="JupyterHub Badge">
</a> [![Download Badge](https://img.shields.io/badge/Download-grey?style=for-the-badge)][download-link]

[download-link]: https://nmfs-opensci.github.io/NMFSHackDays-2025/topics-2025/2025-dask/Dask_gateway.ipynb
[colab-link]: https://colab.research.google.com/github/nmfs-opensci/nmfshackdays-2025/blob/main/topics-2025/2025-dask/Dask_gateway.ipynb
[jupyter-link]: https://nmfs-openscapes.2i2c.cloud/hub/user-redirect/lab?fromURL=https://raw.githubusercontent.com/nmfs-opensci/nmfshackdays-2025/main/topics-2025/2025-dask/Dask_gateway.ipynb

>📘 Learning Objectives
>
> 1. Set up a Dask Gateway cluster
> 2. Learn about the constraints for the Gateway clusters

## Overview

In this notebook, I set up a Dask Gateway cluster. This is a cluster set up on new pods (new virtual machines). I can set up the size and number of the new pods. I can start my notebook with a small pod (4CPU, 2 Gb) and run my tasks on larger pods (many CPU + RAM) or a lot of little pods (1 CPU + less RAM), which is the default. The nmfs-openscapes Jupyter Hub is set-up with Dask Gateway to allow this; it is a common but not a default feature of Jupyter Hubs.

| Feature                 | `LocalCluster`      | `DaskGateway`          |
|------------------------|--------------------------|-------------------------------|
| Runs in your notebook? | ✅ Yes                    | ❌ No (runs in new pods)       |
| Uses multiple pods?    | ❌ No                     | ✅ Yes (scheduler + workers)   |
| Scales beyond pod?     | ❌ No                     | ✅ Yes                         |
| Can use files in /home |  ✅ Yes                   | ❌ No                          |

### Important note on the image

The new pods need to use the same image as the one you started the notebook. Using a small an image as possible will reduce the set-up time since each new pod need to pull down the image into the cluster pod. For this tutorial, I am using

`openscapes/python:07980b9`

using the "Other" option.

### Important note on file access

Since the **Dask Gateway pods (scheduler + workers)** are separate from your notebook pod, they **do not have access to files on local paths like `/home`**.

If your code references a file like `ds = xr.open_dataset("file.json")`, it will fail when `ds["sst"].mean().compute()` is called — because the workers can’t see that file (which is in `\home`). `ds` is lazy and it needs the information in "file.json" to know where the underlying data files are (that it needs to read).

The error will say "FileNotFoundError" and "RuntimeError: Error during deserialization of the task graph. This frequently occurs if the Scheduler and Client have different environments."

## Set up the functions

In [10]:
# import needed modules
import time, random

# define our functions
def inc(x):
    time.sleep(random.random())
    return x + 1

def dec(x):
    time.sleep(random.random())
    return x - 1

def add(x, y):
    time.sleep(random.random())
    return x + y

## A task that takes a long time

20 of these tasks takes about 6 seconds on our 4 CPU pod run in a parallel way with a local Dask cluster. We are going to run 1000 of the tasks, which would take 5-10 minutes.

In [10]:
%%time

# a sequential example with no parallelization
results = []
for x in range(20):
    result = inc(x)
    result = dec(result)
    results.append(result)

print(results)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
CPU times: user 1.21 s, sys: 78.5 ms, total: 1.29 s
Wall time: 18.3 s


## Set up a Dask Gateway cluster

In [11]:
from dask_gateway import Gateway
gateway = Gateway()  # instantiate Dask gateway 
options = gateway.cluster_options()
cluster = gateway.new_cluster(options)
cluster.adapt(minimum=4, maximum=30)

Look at the cluster options. The r5.4xlarge instance is 16 vCPUs, 128.0 GiB.

In [12]:
# look at the cluster options
options

We can open the cluster in the Dask dashboard by clicking on the url.

In [13]:
# look at the cluster
cluster

Open a client run tasks and then close the client. Setting up a Dask Gateway cluster takes awhile (getting those images into the pods). The overhead only makes sense if you are doing big tasks. Also smaller images can speed up set up since you can pull those faster into the pods. The work took 2 minutes and most of that was the set-up time.

In [14]:
%%time
# Set up a client for work
client = cluster.get_client()

results = []
for x in range(1000):
    result = client.submit(inc, x)
    result = client.submit(dec, result)
    results.append(result)

results = client.gather(results)
print(results)
client.close()

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

In [16]:
# When we are done we can close our dask cluster
cluster.close()

## Summary

Dask set up 16 workers. We gave each worker 1 CPU and r5.4xlarge instances (the default node size our our Jupyter Hub) is 16 vCPUs, 128.0 GiB.

![](workers.png)

This notebook used the default worker size but we can change the workers as follows.

```
options = gateway.cluster_options()
options.worker_cores = 2
options.worker_memory = "8GiB"
```